In [2]:
import os
import psycopg2
import pandas as pd
import warnings

warnings.filterwarnings("ignore")


file_path = "insurance_claims.csv"  
insurance_df = pd.read_csv(file_path)


insurance_df = insurance_df.loc[:, ~insurance_df.columns.str.contains('^Unnamed')]


insurance_df.columns = insurance_df.columns.str.strip()


insurance_df.columns = [col.replace(" ", "_").replace("(", "").replace(")", "").replace("%", "pct")
                        for col in insurance_df.columns]


def infer_sql_type(dtype):
    if pd.api.types.is_integer_dtype(dtype):
        return "INT"
    elif pd.api.types.is_float_dtype(dtype):
        return "FLOAT"
    elif pd.api.types.is_bool_dtype(dtype):
        return "BOOLEAN"
    elif pd.api.types.is_datetime64_any_dtype(dtype):
        return "TIMESTAMP"
    else:
        return "TEXT"


conn = psycopg2.connect(
    dbname="postgres",
    user="postgres",
    password="root", 
    host="localhost",
    port="5432"
)
cur = conn.cursor()


table_name = "insurance_claims"
columns = insurance_df.dtypes
sql_columns = ",\n  ".join([f'"{col}" {infer_sql_type(dtype)}' for col, dtype in columns.items()])

create_stmt = f"""
CREATE TABLE "{table_name}" (
  {sql_columns}
);
"""

cur.execute(f'DROP TABLE IF EXISTS "{table_name}" CASCADE;')
cur.execute(create_stmt)
conn.commit()
print("Table recreated with schema:")
print(create_stmt)


columns_list = list(insurance_df.columns)
placeholders = ', '.join(['%s'] * len(columns_list))
quoted_cols = ', '.join([f'"{col}"' for col in columns_list])

insert_stmt = f'INSERT INTO "{table_name}" ({quoted_cols}) VALUES ({placeholders})'

for _, row in insurance_df.iterrows():
    row_values = [None if pd.isna(val) else val for val in row[columns_list]]
    cur.execute(insert_stmt, tuple(row_values))

conn.commit()
print("Data inserted successfully")


df_from_db = pd.read_sql(f'SELECT * FROM "{table_name}" LIMIT 5', conn)
print("Data fetched from DB:")
print(df_from_db.head())

cur.close()
conn.close()


Table recreated with schema:

CREATE TABLE "insurance_claims" (
  "months_as_customer" INT,
  "age" INT,
  "policy_number" INT,
  "policy_bind_date" TEXT,
  "policy_state" TEXT,
  "policy_csl" TEXT,
  "policy_deductable" INT,
  "policy_annual_premium" FLOAT,
  "umbrella_limit" INT,
  "insured_zip" INT,
  "insured_sex" TEXT,
  "insured_education_level" TEXT,
  "insured_occupation" TEXT,
  "insured_hobbies" TEXT,
  "insured_relationship" TEXT,
  "capital-gains" INT,
  "capital-loss" INT,
  "incident_date" TEXT,
  "incident_type" TEXT,
  "collision_type" TEXT,
  "incident_severity" TEXT,
  "authorities_contacted" TEXT,
  "incident_state" TEXT,
  "incident_city" TEXT,
  "incident_location" TEXT,
  "incident_hour_of_the_day" INT,
  "number_of_vehicles_involved" INT,
  "property_damage" TEXT,
  "bodily_injuries" INT,
  "witnesses" INT,
  "police_report_available" TEXT,
  "total_claim_amount" INT,
  "injury_claim" INT,
  "property_claim" INT,
  "vehicle_claim" INT,
  "auto_make" TEXT,
  "auto